<a href="https://colab.research.google.com/github/busycaesar/LLM_Eval/blob/Master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets anthropic tqdm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00


## 0. Configuration

In [ ]:
MODEL = "claude-sonnet-5"
DATASET = "cais/mmlu"
SAMPLE_SIZE = 10

# Parallel requests
MAX_WORKERS = 8



# To be moved
SUBSET = "all"
SPLIT = "test"
SEED = 0

## 1. Load Model

In [ ]:
from google.colab import userdata
from anthropic import Anthropic

client = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

## 2. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET, SUBSET, split=SPLIT)

if SAMPLE_SIZE:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_SIZE, len(ds))))

rows = [dict(r) for r in ds]
print(f"{len(rows)} items loaded")
print(rows[0])

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.50MB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  408kB            

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

10 items loaded
{'question': 'A landlord was the owner of a large, high-rise apartment building in a Midwestern city. On June 1, 2007, two tenants took possession of a three- bedroom apartment in the landlord\'s building under a three-year lease at a rental of $1,200 per month. Their lease (as all other leases given by the landlord) contained the following provisions:"The term of this lease shall be three years from the date hereof as long as all the agreements herein shall be faithfully performed. "The two tenants lived in the apartment for two years. On June 10, 2009, however, a fire destroyed the apartment building. As a result, all the apartments in the building were rendered uninhabitable. After the two tenants were dispossessed from their apartment, the landlord brought suit against them to recover the rent due for the balance of the lease. The two tenants claim that they are no longer liable for rent or any other obligations under the lease. The landlord \x80\x94 tenants leaseho

## 3. Prompt template

In [ ]:
LETTERS = ["A", "B", "C", "D"]

def build_prompt(row):
    choices = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))
    return (
        "Answer the following multiple choice question.\n\n"
        f"Question: {row['question']}\n\n"
        f"{choices}\n\n"
        "Reply with only the letter of the correct answer."
    )

print(build_prompt(rows[0]))

Answer the following multiple choice question.

Question: A landlord was the owner of a large, high-rise apartment building in a Midwestern city. On June 1, 2007, two tenants took possession of a three- bedroom apartment in the landlord's building under a three-year lease at a rental of $1,200 per month. Their lease (as all other leases given by the landlord) contained the following provisions:"The term of this lease shall be three years from the date hereof as long as all the agreements herein shall be faithfully performed. "The two tenants lived in the apartment for two years. On June 10, 2009, however, a fire destroyed the apartment building. As a result, all the apartments in the building were rendered uninhabitable. After the two tenants were dispossessed from their apartment, the landlord brought suit against them to recover the rent due for the balance of the lease. The two tenants claim that they are no longer liable for rent or any other obligations under the lease. The landlo

## 4. Model call with retry and answer extraction function

In [ ]:
import re, time

def call_model(prompt, retries=4):
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=16,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(b.text for b in resp.content if b.type == "text")
        except Exception as e:
            if attempt == retries - 1:
                return f"ERROR: {e}"
            time.sleep(2 ** attempt)

def extract_letter(text):
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None

## 5. Run the eval

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import pd

def evaluate_one(row):
    raw_response = call_model(build_prompt(row))
    prediction = extract_letter(raw_response)
    correct_answer = LETTERS[row["answer"]]

    return {
        "subject": row["subject"],
        "question": row["question"],
        "gold": correct_answer,
        "prediction": prediction,
        "raw_response": raw_response,
        "correct": prediction == correct_answer,
    }

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    results = list(tqdm(ex.map(evaluate_one, rows), total=len(rows)))

df = pd.DataFrame(results)
df.head()

  0%|          | 0/10 [00:00<?, ?it/s]

,subject,question,gold,prediction,raw_response,correct
0,professional_law,"A landlord was the owner of a large, high-rise...",C,C,C,True
1,sociology,Weber (1919) said that the state's monopoly of...,B,B,B,True
2,elementary_mathematics,Which expression is equivalent to 5(4x + 3) — 2x?,A,A,A,True
3,high_school_macroeconomics,Congress has embarked on another round of expa...,D,D,D,True
4,professional_law,"Immediately after a shooting incident, the pol...",D,D,D,True


## 6. Score and aggregate

In [ ]:
accuracy = df["correct"].mean()
unparsed = df["prediction"].isna().sum()
errors = df["raw_response"].str.startswith("ERROR:").sum()

by_subject = (
    df.groupby("subject")["correct"]
      .agg(accuracy="mean", n="count")
      .sort_values("accuracy", ascending=False)
)

Model:     claude-sonnet-5
Dataset:   cais/mmlu / all / test
Items:     10
Accuracy:  1.000
Unparsed:  0
API errors: 0


,accuracy,n
subject,,
elementary_mathematics,1.0,1
high_school_biology,1.0,1
high_school_computer_science,1.0,1
high_school_geography,1.0,1
high_school_macroeconomics,1.0,1
nutrition,1.0,1
professional_law,1.0,2
sociology,1.0,1
world_religions,1.0,1


## 7. Save results

In [ ]:
slug = MODEL.replace("/", "_")
df.to_csv(f"results_{slug}.csv", index=False)
by_subject.to_csv(f"by_subject_{slug}.csv")

print(f"Model:     {MODEL}")
print(f"Dataset:   {DATASET}")
print(f"Items:     {len(df)}")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Unparsed:  {unparsed}")
print(f"API errors: {errors}")

{'model': 'claude-sonnet-5', 'dataset': 'cais/mmlu/all/test', 'items': 10, 'accuracy': 1.0, 'unparsed': 0, 'api_errors': 0, 'seed': 0}
